In [1]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

In [2]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 3, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 40.7143,
	"longitude": -74.006,
	"hourly": ["temperature_2m", "rain", "visibility", "wind_speed_80m", "wind_direction_80m", "temperature_80m", "soil_temperature_18cm", "soil_moisture_3_to_9cm"],
	"timezone": "America/New_York",
}

In [3]:
responses = openmeteo.weather_api(url, params = params)
print(responses)
print(f"\n{type(responses)}")


<class 'list'>


In [6]:
print(responses[0])

In [7]:
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 40.71033477783203°N -73.99308013916016°E
Elevation: 51.0 m asl
Timezone: b'America/New_York'b'GMT-4'
Timezone difference to GMT+0: -14400s


In [9]:
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_rain = hourly.Variables(1).ValuesAsNumpy()
hourly_visibility = hourly.Variables(2).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_direction_80m = hourly.Variables(4).ValuesAsNumpy()
hourly_temperature_80m = hourly.Variables(5).ValuesAsNumpy()
hourly_soil_temperature_18cm = hourly.Variables(6).ValuesAsNumpy()
hourly_soil_moisture_3_to_9cm = hourly.Variables(7).ValuesAsNumpy()

In [10]:
print(hourly_temperature_2m)

[26.657501 25.9575   25.657501 25.2075   24.9575   24.7575   24.4575
 24.8575   25.657501 26.7575   27.8575   28.8575   29.8075   31.2075
 32.607502 33.5575   33.3075   32.357502 30.657501 26.2075   27.6075
 26.407501 24.7575   24.2075   23.7075   23.4575   23.407501 23.407501
 23.157501 22.8075   22.5575   23.0575   23.8575   24.907501 26.1075
 27.2575   29.1075   31.5575   32.5575   32.8075   32.9575   32.2575
 32.506374 31.755249 30.604126 30.003    28.403    27.203001 26.303
 25.003    24.603    24.503    24.203001 23.803    23.603    23.803
 24.703001 26.403    28.203001 30.203001 31.403    32.503    33.403
 34.102997 34.102997 33.703    32.903    31.803    31.303    30.303
 29.603    29.003    28.103    27.603    27.203001 26.703001 26.103
 25.703001 25.303    25.503    26.503    27.903    28.403    28.403
 28.403    29.303    29.603    31.103    31.903002 32.003    31.803
 31.403    30.603    30.203001 29.803    29.103    27.703001 26.903
 26.103    25.403    24.803    24.503   

In [31]:
print(len(hourly_temperature_2m))

168


In [11]:
hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["rain"] = hourly_rain
hourly_data["visibility"] = hourly_visibility
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
hourly_data["temperature_80m"] = hourly_temperature_80m
hourly_data["soil_temperature_18cm"] = hourly_soil_temperature_18cm
hourly_data["soil_moisture_3_to_9cm"] = hourly_soil_moisture_3_to_9cm

hourly_dataframe = pd.DataFrame(data = hourly_data)

In [12]:
hourly_dataframe

,date,temperature_2m,rain,visibility,wind_speed_80m,wind_direction_80m,temperature_80m,soil_temperature_18cm,soil_moisture_3_to_9cm
0,2026-08-07 00:00:00-04:00,26.657501,0.0,16900.0,20.523157,232.124954,27.729000,28.496500,0.315
1,2026-08-07 01:00:00-04:00,25.957500,0.0,16300.0,19.559633,263.659912,27.229000,28.296499,0.320
2,2026-08-07 02:00:00-04:00,25.657501,0.0,14800.0,14.255272,315.000092,26.828999,28.046499,0.320
3,2026-08-07 03:00:00-04:00,25.207500,0.0,14400.0,13.004921,265.236450,26.479000,27.846500,0.320
4,2026-08-07 04:00:00-04:00,24.957500,0.0,13800.0,10.086427,267.954651,26.129000,27.596500,0.320
...,...,...,...,...,...,...,...,...,...
163,2026-08-13 19:00:00-04:00,26.503000,0.0,24140.0,29.055367,303.886993,27.629000,28.896500,0.301
164,2026-08-13 20:00:00-04:00,25.802999,0.0,24140.0,27.804029,318.674591,26.528999,28.846500,0.301
165,2026-08-13 21:00:00-04:00,25.103001,0.0,24140.0,27.943514,321.801270,25.328999,28.696501,0.301
166,2026-08-13 22:00:00-04:00,24.353001,0.0,24140.0,28.075384,319.159729,24.078999,28.446501,0.302


In [13]:
hourly_dataframe["date"].head(10)

0   2026-08-07 00:00:00-04:00
1   2026-08-07 01:00:00-04:00
2   2026-08-07 02:00:00-04:00
3   2026-08-07 03:00:00-04:00
4   2026-08-07 04:00:00-04:00
5   2026-08-07 05:00:00-04:00
6   2026-08-07 06:00:00-04:00
7   2026-08-07 07:00:00-04:00
8   2026-08-07 08:00:00-04:00
9   2026-08-07 09:00:00-04:00
Name: date, dtype: datetime64[s, America/New_York]

In [26]:
hourly_dataframe.columns

Index(['date', 'temperature_2m', 'rain', 'visibility', 'wind_speed_80m',
       'wind_direction_80m', 'temperature_80m', 'soil_temperature_18cm',
       'soil_moisture_3_to_9cm'],
      dtype='str')

In [28]:
hourly_dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype                          
---  ------                  --------------  -----                          
 0   date                    168 non-null    datetime64[s, America/New_York]
 1   temperature_2m          168 non-null    float32                        
 2   rain                    168 non-null    float32                        
 3   visibility              168 non-null    float32                        
 4   wind_speed_80m          168 non-null    float32                        
 5   wind_direction_80m      168 non-null    float32                        
 6   temperature_80m         168 non-null    float32                        
 7   soil_temperature_18cm   168 non-null    float32                        
 8   soil_moisture_3_to_9cm  168 non-null    float32                        
dtypes: datetime64[s, America/New_York](1), float32(8)
memor

In [30]:
hourly_md = hourly_dataframe.to_markdown(buf="results.md")